In [1]:
import re
import pandas as pd
import numpy as np

def converter_expressao_para_regex(expressao_booleana):
    """Conversor automático de expressões (A OU B) E (C OU D) para Regex Lookahead"""
    if not isinstance(expressao_booleana, str) or not expressao_booleana.strip():
        return ""

    # 1. Encontra tudo o que está dentro dos parênteses (...)
    # Isso isola os grupos que estão separados por " E "
    grupos_parenteses = re.findall(r"\((.*?)\)", expressao_booleana)

    # Se a estrutura não tiver parênteses, tenta quebrar direto pelo " E "
    if not grupos_parenteses:
        grupos_parenteses = expressao_booleana.split(" E ")

    lookaheads = []

    for grupo in grupos_parenteses:
        # Quebra os termos internos pelo " OU "
        termos = [termo.strip() for termo in grupo.split(" OU ") if termo.strip()]

        if not termos:
            continue

        # Ordena do maior termo para o menor (evita que 'DP' engula 'Defensoria Pública')
        termos_ordenados = sorted(termos, key=len, reverse=True)

        # Escapa caracteres especiais que possam existir no texto (como pontos ou interrogações)
        termos_escapados = "|".join(
            [re.escape(termo) for termo in termos_ordenados]
        )

        # Monta o Lookahead para este grupo específico garantindo as fronteiras de palavra (\b)
        # O .*? garante que ele ache o termo em qualquer parte do texto longo
        lookahead_grupo = f"(?=.*?(?:{termos_escapados}))"
        lookaheads.append(lookahead_grupo)

    # Une todos os lookaheads. Para a Regex dar MATCH, TODOS os grupos precisam ser verdadeiros (Lógica "E")
    #regex_final = "|".join(lookaheads)
    regex_final = "".join(lookaheads)
    return regex_final


In [2]:
# 1. Carrega o seu CSV (ajustando para o encoding padrão que costuma vir do Excel)
df_temas = pd.read_csv(
    r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Palavras chaves temas do STF - Leonardo.csv", encoding="utf-8"
)
df_temas_2 = pd.read_csv(
    r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Palavras chaves temas do STJ - Patrícia.csv", encoding="utf-8"
)

# 2. Aplica a conversão automática criando a nova coluna 'REGEX'
df_temas["REGEX"] = df_temas["PALAVRAS-CHAVES"].apply(
    converter_expressao_para_regex
)
df_temas_2["REGEX"] = df_temas_2["PALAVRAS-CHAVES"].apply(
    converter_expressao_para_regex
)
df_temas = pd.concat([df_temas, df_temas_2], ignore_index=True)
# 3. Salva o resultado em um novo arquivo CSV para você usar
df_temas.to_csv("Temas_STF_com_Regex_Parcial.csv", index=False, encoding="utf-8")

print("Sucesso! O arquivo 'Temas_STF_com_Regex_Parcial.csv' foi gerado com as Regex.")

Sucesso! O arquivo 'Temas_STF_com_Regex_Parcial.csv' foi gerado com as Regex.


In [3]:
import re
import pandas as pd

def aplicar_regex_no_dataframe(df, df_regex):
    """Aplica as expressões regulares de df_regex no df e lista os temas encontrados."""
    # Inicializa a coluna com listas vazias para cada linha
    df = df.copy()
    df.loc[:, "temas_encontrados"] = [[] for _ in range(len(df))]

    for _, row in df_regex.iterrows():
        regex = row["REGEX"]
        tema_codigo = row["TEMA CÓDIGO"]  # Nome exato da coluna do seu CSV

        # Pula se a regex estiver vazia
        if regex=="NAN":
            continue

        # Cria uma máscara booleana para as linhas que dão match
        matches = df["inteiro_teor_lematizado"].str.contains(
            regex, flags=re.IGNORECASE, regex=True, na=False
        )
        total_matches = df["inteiro_teor_lematizado"].str.contains(
            regex, flags=re.IGNORECASE, regex=True, na=False
        ).value_counts()
        print("Total matches:", total_matches)
        # Para as linhas onde deu Match, adiciona o código do tema na lista
        df.loc[matches, "temas_encontrados"] = df.loc[
            matches, "temas_encontrados"
        ].apply(lambda x: x + [tema_codigo])

    return df

In [5]:
df_lematizado = pd.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\dataset_enriquecido_10062026_lematizado.parquet")
df_temas = pd.read_csv(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification-1\data\Temas_STF_com_Regex_Parcial.csv")

In [6]:
df_lematizado_sample = df_lematizado.sample(frac=0.01, random_state=42)
df_sample = df_lematizado_sample.iloc[[0]]
df_tema = df_temas.iloc[[4]]

In [ ]:
df_temas["REGEX"].fillna("NAN",inplace=True)
df_lematizado_sample_with_regex = aplicar_regex_no_dataframe(df_sample, df_temas)

C:\Users\lfmelo\AppData\Local\Temp\ipykernel_19760\346091390.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_temas["REGEX"].fillna("NAN",inplace=True)


Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matches: inteiro_teor_lematizado
False    1
Name: count, dtype: int64
Total matche